# Phase 11: Robustness (RQ3) and Reliability Analysis (RQ4)
**Objective**: Evaluate the baseline **and** the proposed model (best checkpoints, all seeds) on the test split under controlled degradation, and analyse whether the estimated uncertainty is informative and whether the fusion weights react to it.

**Conditions** (identical corrupted inputs for every model and seed):
- `clean`
- `missing_<modality>`: every utterance of that modality is **flagged** unusable (the learned missing-token path used for failed clips; PoE models also give it gate 0)
- `dead_<modality>`: the modality's features are all zero but **not flagged** (a silently dead sensor: only the model's own uncertainty can reveal it)
- `noise_<modality>_<s>`: Gaussian noise with per-dimension std `s x` (train-set per-dimension std of that modality), levels s in {0.5, 1, 2, 4}

**RQ3**: macro-F1 / accuracy / minority F1 per condition, retention relative to clean, paired proposed-baseline differences with Holm correction across conditions.
**RQ4**: (a) AUROC of the normalised uncertainty for detecting each modality head's own errors, (b) change of the mean fusion weight and mean uncertainty of the corrupted modality, baseline vs proposed, (c) distributions of `u` and `w` clean vs degraded.

In [ ]:
# Required Imports
import os
import sys
import json
import copy
import importlib
import numpy as np
import pandas as pd
import torch

print("Libraries imported successfully.")

In [ ]:
# 1. Configuration and prerequisites (Phases 9-10)
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
config = json.load(open(os.path.join(BASE_DIR, "config.json")))
context_config = json.load(open(os.path.join(BASE_DIR, "context_config.json")))
RESULTS_DIR = config["paths"]["results_dir"]
for fn in ("phase9_run_index.json", "phase10_test_results.json"):
    assert os.path.exists(os.path.join(RESULTS_DIR, fn)), f"Missing results/{fn} -- run Phases 9-10 first."
run_index = json.load(open(os.path.join(RESULTS_DIR, "phase9_run_index.json")))
p10 = json.load(open(os.path.join(RESULTS_DIR, "phase10_test_results.json")))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)
from meld_eval import mean_std, paired_stats, holm, auroc_safe
from meld_models import load_split_dataset
from meld_context import make_loader, MELDDialogueDataset, load_split_features
from meld_train import load_best_model, evaluate_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMOTIONS, IN_DIMS = config["emotions"], context_config["in_dims"]
C = len(EMOTIONS)
BASELINE, PROPOSED, SEEDS = p10["baseline"], p10["proposed"], p10["seeds"]
MODS = ["text", "audio", "video"]
P11_DIR = os.path.join(RESULTS_DIR, "phase11")
os.makedirs(P11_DIR, exist_ok=True)

def env_list(name, default, cast):
    return [cast(x) for x in os.environ.get(name, default).split(",") if x.strip()]
NOISE_LEVELS = env_list("MELD_P11_NOISE", "0.5,1,2,4", float)

CONDITIONS = [("clean", "clean", None, None)]
CONDITIONS += [(f"missing_{m}", "missing", m, None) for m in MODS]
CONDITIONS += [(f"dead_{m}", "dead", m, None) for m in MODS]
CONDITIONS += [(f"noise_{m}_{s:g}", "noise", m, s) for m in MODS for s in NOISE_LEVELS]
print(f"Models: baseline='{BASELINE}', proposed='{PROPOSED}' | seeds {SEEDS} | {len(CONDITIONS)} conditions | noise levels {NOISE_LEVELS}")

In [ ]:
# 2. Test data (arrays) and the per-dimension train-set std used to scale the noise
test_df, _ = load_split_dataset(BASE_DIR, "test", IN_DIMS, EMOTIONS, context_config["speaker_mode"])
manifest = pd.read_csv(os.path.join(BASE_DIR, "data", "context_manifest_test.csv"))
base_feats, base_valid = load_split_features(os.path.join(BASE_DIR, "features"), "test", IN_DIMS)
tr_feats, tr_valid = load_split_features(os.path.join(BASE_DIR, "features"), "train", IN_DIMS)
train_std = {m: np.maximum(tr_feats[m][tr_valid[m]].std(axis=0), 1e-6).astype(np.float32) for m in MODS}

def build_condition(idx, kind, modality, level):
    """Returns (dataset, feats, valid). Only the targeted modality is altered; noise realisation depends only on the condition index."""
    feats, valid = dict(base_feats), {m: v.copy() for m, v in base_valid.items()}
    if kind == "missing":
        valid[modality] = np.zeros_like(valid[modality])
    elif kind == "dead":
        feats[modality] = np.zeros_like(base_feats[modality])          # silent: validity flag stays True
    elif kind == "noise":
        rs = np.random.RandomState(10_000 + idx)
        feats[modality] = (base_feats[modality] + level * train_std[modality] * rs.randn(*base_feats[modality].shape)).astype(np.float32)
    ds = MELDDialogueDataset(test_df, manifest, feats, valid, EMOTIONS, speaker_mode=context_config["speaker_mode"])
    return ds, feats, valid
print("Condition builder defined.")

In [ ]:
# 3. Load the best checkpoints of the two compared models (all seeds)
models = {}
for rec in run_index:
    if rec["name"] in (BASELINE, PROPOSED):
        models[(rec["name"], rec["seed"])], _ = load_best_model(rec["run_dir"], device)
assert len(models) == 2 * len(SEEDS)
print(f"Loaded {len(models)} models.")

In [ ]:
# 4. Evaluate every condition x model x seed
rows, clean_arr, dist_u, dist_w = [], {}, {}, {}
max_level = max(NOISE_LEVELS)
keep_dist = {"clean"} | {f"missing_{m}" for m in MODS} | {f"noise_{m}_{max_level:g}" for m in MODS}
for idx, (cname, kind, mod, level) in enumerate(CONDITIONS):
    ds, _, _ = build_condition(idx, kind, mod, level)
    loader = make_loader(ds, batch_size=32, shuffle=False, seed=0)
    for (name, seed), model in models.items():
        met, arr = evaluate_model(model, loader, device, EMOTIONS, eval_seed=0)
        row = {"condition": cname, "kind": kind, "modality": mod, "level": level, "model": name, "seed": seed,
               "accuracy": met["accuracy"], "weighted_f1": met["weighted_f1"], "macro_f1": met["macro_f1"],
               "f1_fear": met["per_class"]["fear"]["f1"], "f1_disgust": met["per_class"]["disgust"]["f1"],
               "tv_shift": float(np.abs(arr["w"] - arr["w_content"]).sum(1).mean() / 2)}
        for i, m in enumerate(MODS):
            row[f"head_acc_{m}"] = float((arr["mod_pred"][:, i] == arr["y_true"]).mean())
            row[f"u_{m}"], row[f"uraw_{m}"], row[f"w_{m}"] = float(arr["u"][:, i].mean()), float(arr["u_raw"][:, i].mean()), float(arr["w"][:, i].mean())
        rows.append(row)
        if cname == "clean":
            clean_arr[(name, seed)] = arr
        if cname in keep_dist:
            dist_u.setdefault(f"u__{cname}__{name}", []).append(arr["u"]); dist_w.setdefault(f"w__{cname}__{name}", []).append(arr["w"])
    sub = pd.DataFrame([r for r in rows if r["condition"] == cname])
    print(f"{cname:18s} macro-F1  " + " | ".join(f"{n}: {g.macro_f1.mean():.4f}" for n, g in sub.groupby("model", sort=False)))
rob = pd.DataFrame(rows)
rob.to_csv(os.path.join(P11_DIR, "robustness_runs.csv"), index=False)
np.savez_compressed(os.path.join(P11_DIR, "reliability_distributions.npz"),
                    **{k: np.concatenate(v) for k, v in {**dist_u, **dist_w}.items()})

In [ ]:
# 5. RQ3: aggregate over seeds, retention and paired proposed-baseline statistics (Holm across degraded conditions)
clean_f1 = rob[rob.condition == "clean"].set_index(["model", "seed"])["macro_f1"]
rob["retention"] = [r.macro_f1 / clean_f1[(r.model, r.seed)] for r in rob.itertuples()]
summary_rows = []
for (cname, name), g in rob.groupby(["condition", "model"], sort=False):
    d = {"condition": cname, "model": name, "kind": g["kind"].iloc[0], "modality": g["modality"].iloc[0], "level": g["level"].iloc[0], "n_seeds": len(g)}
    for k in ("accuracy", "macro_f1", "weighted_f1", "f1_fear", "f1_disgust", "retention"):
        d[f"{k}_mean"], d[f"{k}_std"] = mean_std(g[k].values)
    summary_rows.append(d)
rob_summary = pd.DataFrame(summary_rows)
rob_summary.to_csv(os.path.join(P11_DIR, "robustness_summary.csv"), index=False)

paired_rows = []
for cname, kind, mod, level in CONDITIONS:
    g = rob[rob.condition == cname]
    a = g[g.model == PROPOSED].sort_values("seed"); b = g[g.model == BASELINE].sort_values("seed")
    for metric in ("macro_f1", "retention"):
        s = paired_stats(a[metric].values, b[metric].values)
        paired_rows.append({"condition": cname, "kind": kind, "modality": mod, "level": level, "metric": metric, **s})
paired_df = pd.DataFrame(paired_rows)
for metric in ("macro_f1", "retention"):
    sel = (paired_df.metric == metric) & (paired_df.condition != "clean")
    paired_df.loc[sel, "p_holm"] = holm(paired_df.loc[sel, "p_value"].values)
paired_df.to_csv(os.path.join(P11_DIR, "robustness_paired.csv"), index=False)

pv = rob_summary.pivot(index="condition", columns="model", values="macro_f1_mean").loc[[c[0] for c in CONDITIONS]]
print("Macro-F1 (mean over seeds) by condition:")
print(pv.round(4).to_string())
print("\nPaired difference in macro-F1 (proposed - baseline), degraded conditions:")
print(paired_df[(paired_df.metric == "macro_f1") & (paired_df.condition != "clean")][["condition", "mean_diff", "ci_low", "ci_high", "p_value", "p_holm"]].round(4).to_string(index=False))

In [ ]:
# 6. RQ4: is the uncertainty informative, and do the fusion weights react to corruption?
rel = {"auroc_u_detects_head_errors_clean": {}, "clean_weight_shift_tv": {}, "corruption_response": []}
for name in (BASELINE, PROPOSED):
    rel["auroc_u_detects_head_errors_clean"][name] = {}
    for i, m in enumerate(MODS):
        vals = [auroc_safe(clean_arr[(name, sd)]["u"][:, i], clean_arr[(name, sd)]["mod_pred"][:, i] != clean_arr[(name, sd)]["y_true"]) for sd in SEEDS]
        rel["auroc_u_detects_head_errors_clean"][name][m] = dict(zip(("mean", "std"), mean_std(vals)))
    rel["clean_weight_shift_tv"][name] = dict(zip(("mean", "std"), mean_std([
        float(np.abs(clean_arr[(name, sd)]["w"] - clean_arr[(name, sd)]["w_content"]).sum(1).mean() / 2) for sd in SEEDS])))

resp_rows = []
for cname, kind, mod, level in CONDITIONS:
    if kind == "clean":
        continue
    g = rob[rob.condition == cname]
    c0 = rob[rob.condition == "clean"]
    per_model = {}
    for name in (BASELINE, PROPOSED):
        a = g[g.model == name].sort_values("seed"); b = c0[c0.model == name].sort_values("seed")
        per_model[name] = {"d_w": a[f"w_{mod}"].values - b[f"w_{mod}"].values, "d_u": a[f"u_{mod}"].values - b[f"u_{mod}"].values,
                           "d_head_acc": a[f"head_acc_{mod}"].values - b[f"head_acc_{mod}"].values}
    s = paired_stats(per_model[PROPOSED]["d_w"], per_model[BASELINE]["d_w"])
    resp_rows.append({"condition": cname, "kind": kind, "modality": mod, "level": level,
                      **{f"{name}_{k}_{st}": v for name in (BASELINE, PROPOSED) for k in ("d_w", "d_u", "d_head_acc")
                         for st, v in zip(("mean", "std"), mean_std(per_model[name][k]))},
                      "dw_prop_minus_base_mean": s["mean_diff"], "dw_ci_low": s["ci_low"], "dw_ci_high": s["ci_high"], "dw_p": s["p_value"]})
resp = pd.DataFrame(resp_rows)
resp["dw_p_holm"] = holm(resp["dw_p"].values)
resp.to_csv(os.path.join(P11_DIR, "reliability_response.csv"), index=False)
rel["corruption_response"] = resp.to_dict("records")

print("AUROC of normalised uncertainty for detecting each modality head's errors (0.5 = uninformative), clean test:")
print(pd.DataFrame({n: {m: f"{v['mean']:.3f} +/- {v['std']:.3f}" for m, v in d.items()} for n, d in rel["auroc_u_detects_head_errors_clean"].items()}).to_string())
print("\nMean change of the fusion weight of the corrupted modality (d_w) and of its uncertainty (d_u):")
cols = ["condition", f"{BASELINE}_d_w_mean", f"{PROPOSED}_d_w_mean", f"{BASELINE}_d_u_mean", f"{PROPOSED}_d_u_mean", "dw_prop_minus_base_mean", "dw_p_holm"]
print(resp[cols].round(4).to_string(index=False))
with open(os.path.join(P11_DIR, "reliability_analysis.json"), "w") as f:
    json.dump(rel, f, indent=2, default=float)

In [ ]:
# 7. Verification suite
results = []
def check(name, cond, detail=""):
    detail = str(detail)
    results.append({"test": name, "passed": bool(cond), "detail": detail})
    print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))

# R1: the clean condition reproduces the Phase-10 test metrics of every run exactly
worst = 0.0
for (name, seed) in models:
    m10 = json.load(open(os.path.join(RESULTS_DIR, "phase10", name, f"seed{seed}", "metrics.json")))
    r = rob[(rob.condition == "clean") & (rob.model == name) & (rob.seed == seed)].iloc[0]
    worst = max(worst, abs(r["macro_f1"] - m10["macro_f1"]), abs(r["accuracy"] - m10["accuracy"]))
check("R1 clean condition reproduces Phase-10 test metrics", worst < 1e-9, f"max |diff|={worst:.1e}")

# R2: noise is applied only to the target modality and has the requested scale
idx = [c[0] for c in CONDITIONS].index(f"noise_audio_{NOISE_LEVELS[-1]:g}")
_, f2, v2 = build_condition(idx, "noise", "audio", NOISE_LEVELS[-1])
ratio = ((f2["audio"] - base_feats["audio"]).std(axis=0) / (NOISE_LEVELS[-1] * train_std["audio"])).mean()
N_, D_ = base_feats["audio"].shape
tol = max(0.03, 5.0 / np.sqrt(2 * (N_ - 1) * D_))          # 5 standard errors of a sample std averaged over D dims (>= 3%)
check(f"R2 noise: only the target modality changes, noise std = level x train-std (+/-{tol:.1%})",
      abs(ratio - 1) < tol and np.array_equal(f2["text"], base_feats["text"]) and np.array_equal(f2["video"], base_feats["video"])
      and all(np.array_equal(v2[m], base_valid[m]) for m in MODS), f"measured/target = {ratio:.4f}")

# R3: missing-modality condition flags only the target modality
_, f3, v3 = build_condition(1, "missing", "text", None)
check("R3 missing: all utterances of the target modality flagged unusable, nothing else changes",
      (~v3["text"]).all() and np.array_equal(v3["audio"], base_valid["audio"]) and np.array_equal(v3["video"], base_valid["video"])
      and all(np.array_equal(f3[m], base_feats[m]) for m in MODS))

# R3b: dead-sensor condition zeroes only the target modality and does NOT flag it
_, f3b, v3b = build_condition(4, "dead", "audio", None)
check("R3b dead: target features are all zero, validity flags untouched, other modalities unchanged",
      not f3b["audio"].any() and all(np.array_equal(v3b[m], base_valid[m]) for m in MODS) and np.array_equal(f3b["text"], base_feats["text"]) and np.array_equal(f3b["video"], base_feats["video"]))

# R4: conditions are reproducible (identical corrupted data for every model/seed)
_, fa, _ = build_condition(idx, "noise", "audio", NOISE_LEVELS[-1])
_, fb, _ = build_condition(idx, "noise", "audio", NOISE_LEVELS[-1])
check("R4 corrupted inputs are reproducible", np.array_equal(fa["audio"], fb["audio"]))

# R5: completeness / finite values
check("R5 one result per condition x model x seed, all finite", len(rob) == len(CONDITIONS) * len(models) and np.isfinite(rob[["accuracy", "macro_f1", "weighted_f1"]].values).all(),
      f"{len(rob)} rows")

# R6: AUROC helper agrees with sklearn and handles single-class input
from sklearn.metrics import roc_auc_score
rs = np.random.RandomState(0); sc, pos = rs.rand(200), rs.randint(0, 2, 200)
check("R6 AUROC helper equals sklearn; NaN if one class only", abs(auroc_safe(sc, pos) - roc_auc_score(pos, sc)) < 1e-12 and np.isnan(auroc_safe(sc, np.zeros(200))))

# R7: the perturbations actually reach the models (some degraded condition differs from clean)
check("R7 degradation has an effect (macro-F1 differs from clean in at least one condition)", (np.abs(rob[rob.condition != "clean"]["macro_f1"].values.reshape(-1, len(models)).mean(1) - rob[rob.condition == "clean"]["macro_f1"].mean()) > 1e-4).any())

# R8: in 'missing' conditions the corrupted modality's own head is degraded relative to clean (uses the missing-token path)
mm = rob[rob.kind == "missing"]
drops = [(rob[(rob.condition == "clean")][f"head_acc_{r.modality}"].mean() - r[f"head_acc_{r.modality}"]) for _, r in mm.iterrows()]
check("R8 missing-modality conditions change the corrupted modality's head accuracy", np.abs(np.array(drops)).max() > 1e-3, f"max change {np.abs(np.array(drops)).max():.3f}")

failed = [r for r in results if not r["passed"]]
with open(os.path.join(RESULTS_DIR, "robustness_verification.json"), "w") as f:
    json.dump({"n_tests": len(results), "n_failed": len(failed), "results": results}, f, indent=2)
assert not failed, f"{len(failed)} verification test(s) failed: {[r['test'] for r in failed]}"
print(f"\nAll {len(results)} verification checks passed.")

In [ ]:
# 8. Save the machine-readable summary consumed by Phase 12
phase11 = {"baseline": BASELINE, "proposed": PROPOSED, "seeds": SEEDS, "noise_levels": NOISE_LEVELS, "modalities": MODS,
           "conditions": [c[0] for c in CONDITIONS], "noise_definition": "Gaussian, std = level x train-set per-dimension std of the modality",
           "missing_definition": "modality flagged unusable for every utterance (learned missing token)",
           "dead_definition": "modality features set to zero for every utterance WITHOUT flagging it unusable (silent sensor failure)",
           "files": {"runs": "phase11/robustness_runs.csv", "summary": "phase11/robustness_summary.csv", "paired": "phase11/robustness_paired.csv",
                     "reliability_response": "phase11/reliability_response.csv", "reliability": "phase11/reliability_analysis.json",
                     "distributions": "phase11/reliability_distributions.npz"}}
with open(os.path.join(RESULTS_DIR, "phase11_results.json"), "w") as f:
    json.dump(phase11, f, indent=2)
print("Phase 11 Experiments and Analysis completed successfully.")